# PAM Zero Coupon Bond on LocalNet

This notebook runs a live PAM zero coupon bond lifecycle on AlgoKit LocalNet.

It covers both:
- the high-level SDK flow to build ACTUS terms, normalize them, and deploy `DAsa`
- the live execution flow through discounted issuance, `IED`, funding, holder claims, and maturity


## Prerequisites

- run `make install-dev` once
- start LocalNet with `make localnet` or `algokit localnet start`
- rerun the notebook from the first cell on a fresh LocalNet when repeating the demo


In [ ]:
import os
import sys
from pathlib import Path

REPO_ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents] if (path / "pyproject.toml").exists()
)
os.chdir(REPO_ROOT)
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

print(f"Repository root: {REPO_ROOT}")


In [ ]:
from algokit_utils import AssetTransferParams, CommonAppCallParams

from d_asa import (
    DAsa,
    DAsaRole,
    enums,
    make_pam_zero_coupon_bond,
    normalize_contract_attributes,
)
from d_asa import constants as cst
from d_asa.artifacts.dasa_client import PrimaryDistributionArgs
from d_asa.localnet import get_latest_timestamp, time_warp
from d_asa.schedule import Cycle
from examples._support import (
    cashflow_rows,
    connect_localnet,
    create_currency,
    create_funded_account,
    format_asset_amount,
    opt_in_asset,
    render_table,
    schedule_rows,
)


In [ ]:
PRINCIPAL = 10_000
MINIMUM_DENOMINATION = 100
ZERO_COUPON_MATURITY = Cycle.parse_cycle("360D")
ZERO_COUPON_DISCOUNT_BPS = 200
ISSUANCE_DELAY = Cycle.parse_cycle("30D")


def day_cycle_seconds(cycle: Cycle) -> int:
    if cycle.unit != "D":
        raise ValueError(f"Only fixed day cycles are used in this notebook, got {cycle}")
    return cycle.count * cst.DAY_2_SEC


In [ ]:
algorand = connect_localnet()
arranger = create_funded_account(algorand, min_algo=50)
bank = create_funded_account(algorand, min_algo=50)
account_manager = create_funded_account(algorand)
primary_dealer = create_funded_account(algorand)
investor = create_funded_account(algorand)

currency = create_currency(algorand, issuer=bank)
opt_in_asset(algorand, account=investor, asset_id=currency.id)

print(f"Currency ASA id: {currency.id}")
print(f"Arranger: {arranger.address}")
print(f"Bank: {bank.address}")
print(f"Investor: {investor.address}")
print(f"Account manager: {account_manager.address}")
print(f"Primary dealer: {primary_dealer.address}")


In [ ]:
current_ts = get_latest_timestamp(algorand.client.algod)
issuance_date = current_ts + day_cycle_seconds(ISSUANCE_DELAY)
maturity_date = issuance_date + day_cycle_seconds(ZERO_COUPON_MATURITY)
discount_amount = PRINCIPAL * ZERO_COUPON_DISCOUNT_BPS // cst.BPS

attrs = make_pam_zero_coupon_bond(
    contract_id=2,
    status_date=current_ts,
    initial_exchange_date=issuance_date,
    maturity_date=maturity_date,
    notional_principal=PRINCIPAL,
    premium_discount_at_ied=discount_amount,
)
normalized = normalize_contract_attributes(
    attrs,
    denomination_asset_id=currency.id,
    denomination_asset_decimals=currency.decimals,
    notional_unit_value=MINIMUM_DENOMINATION,
    secondary_market_opening_date=issuance_date,
    secondary_market_closure_date=maturity_date + cst.DAY_2_SEC,
)
bank_funding_amount = normalized.terms.notional_principal - normalized.terms.initial_exchange_amount
assert [entry.event_type for entry in normalized.schedule] == ["IED", "MD"]

issue_price_label = format_asset_amount(
    normalized.terms.initial_exchange_amount,
    decimals=currency.decimals,
    unit_name=currency.unit_name,
)
principal_label = format_asset_amount(
    normalized.terms.notional_principal,
    decimals=currency.decimals,
    unit_name=currency.unit_name,
)
discount_label = format_asset_amount(
    bank_funding_amount,
    decimals=currency.decimals,
    unit_name=currency.unit_name,
)
print(f"Issue price: {issue_price_label}")
print(f"Principal at maturity: {principal_label}")
print(f"Discount reserve: {discount_label}")
print(render_table(schedule_rows(normalized, currency_decimals=currency.decimals)))


In [ ]:
app = DAsa.deploy_configured(
    algorand=algorand,
    arranger=arranger,
    normalized=normalized,
    prospectus_url="Notebook example: PAM zero coupon bond",
)
arranger_role = app.arranger(arranger)
arranger_role.assign_role(DAsaRole.ACCOUNT_MANAGER, account_manager.address)
arranger_role.assign_role(DAsaRole.PRIMARY_DEALER, primary_dealer.address)

app.account_manager(account_manager).open_account(
    holding_address=investor.address,
    payment_address=investor.address,
)

state_after_config = app.contract.get_state()
assert state_after_config.status == enums.STATUS_PENDING_IED
print(state_after_config)


## Primary Distribution

The next cell uses `app.raw_client.new_group()` so the investor payment and the
`primary_distribution` app call stay atomic. That grouped flow is not yet wrapped
by a higher-level SDK helper.


In [ ]:
if bank_funding_amount > 0:
    algorand.send.asset_transfer(
        AssetTransferParams(
            asset_id=currency.id,
            amount=bank_funding_amount,
            receiver=app.app_address,
            sender=bank.address,
            signer=bank.signer,
        )
    )

algorand.send.asset_transfer(
    AssetTransferParams(
        asset_id=currency.id,
        amount=normalized.terms.initial_exchange_amount,
        receiver=investor.address,
        sender=bank.address,
        signer=bank.signer,
    )
)

investor_purchase_txn = algorand.create_transaction.asset_transfer(
    AssetTransferParams(
        asset_id=currency.id,
        amount=normalized.terms.initial_exchange_amount,
        receiver=app.app_address,
        sender=investor.address,
        signer=investor.signer,
    )
)
(
    app.raw_client.new_group()
    .add_transaction(investor_purchase_txn, investor.signer)
    .primary_distribution(
        PrimaryDistributionArgs(
            holding_address=investor.address,
            units=normalized.terms.total_units,
        ),
        params=CommonAppCallParams(
            sender=primary_dealer.address,
            signer=primary_dealer.signer,
        ),
    )
    .send()
)

raw_position = app.account(investor).get_raw_position()
assert raw_position.reserved_units == normalized.terms.total_units
print(raw_position)
print(
    "Contract currency balance:",
    format_asset_amount(
        algorand.asset.get_account_information(app.app_address, currency.id).balance,
        decimals=currency.decimals,
        unit_name=currency.unit_name,
    ),
)


In [ ]:
pending_state = app.contract.get_state()
assert pending_state.status == enums.STATUS_PENDING_IED

ied_entry = normalized.schedule[0]
time_warp(ied_entry.scheduled_time + cst.DAY_2_SEC, algorand=algorand)
arranger_role.execute_ied()

active_state = app.contract.get_state()
actualized_position = app.account(investor).get_actualized_position()
assert active_state.status == enums.STATUS_ACTIVE
assert actualized_position.units == normalized.terms.total_units

print(active_state)
print(actualized_position)
print("Next due event:", app.contract.get_next_due_event())


In [ ]:
holding = app.account(investor)
cashflows = []

for entry in normalized.schedule[1:]:
    time_warp(entry.scheduled_time, algorand=algorand)
    funding = arranger_role.fund_due_cashflows(max_event_count=1)
    claim = holding.claim()

    assert funding.processed_events == 1
    assert claim.total_amount == funding.total_funded

    cashflows.append(
        {
            "event_id": entry.event_id,
            "event_type": entry.event_type,
            "scheduled_time": entry.scheduled_time,
            "funded": funding.total_funded,
            "interest_claimed": claim.interest_amount,
            "principal_claimed": claim.principal_amount,
            "total_claimed": claim.total_amount,
            "claim_timestamp": claim.timestamp,
        }
    )

print(render_table(cashflow_rows(cashflows, currency=currency)))


In [ ]:
final_state = app.contract.get_state()
final_position = holding.get_actualized_position()
total_interest_claimed = sum(int(item["interest_claimed"]) for item in cashflows)
total_principal_claimed = sum(int(item["principal_claimed"]) for item in cashflows)

assert final_state.status == enums.STATUS_ENDED
assert final_state.event_cursor == len(normalized.schedule)
assert total_interest_claimed == 0
assert total_principal_claimed == normalized.terms.notional_principal
assert final_position.units == normalized.terms.total_units

print(final_state)
print(final_position)
paid_label = format_asset_amount(
    normalized.terms.initial_exchange_amount,
    decimals=currency.decimals,
    unit_name=currency.unit_name,
)
received_label = format_asset_amount(
    total_interest_claimed + total_principal_claimed,
    decimals=currency.decimals,
    unit_name=currency.unit_name,
)
print(f"Investor paid {paid_label}")
print(f"Investor received {received_label}")
